# Fashion-MNIST Image Classification with a Convolutional Neural Network

This notebook builds, trains, and evaluates a CNN that classifies grayscale clothing images from the Fashion-MNIST dataset into 10 categories.

**Goal:** classify each 28x28 grayscale image into one of 10 classes (T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot).

**Approach:** a custom 3-block CNN trained from scratch (no transfer learning, since the images are small, single-channel, and the dataset is large enough for a small CNN to learn good features on its own).

Result achieved in this run: **91.18% test accuracy**, **0.2382 test loss**, training stopped early at epoch 5 by `EarlyStopping`.

In [ ]:
# Imports
# numpy/matplotlib for arrays and plots, tensorflow/keras for the model itself,
# sklearn for the confusion matrix + classification report, seaborn just for the heatmap styling.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)

## 1. Loading the Dataset

Fashion-MNIST ships directly with Keras, so there is no manual download step. It has 60,000 training images and 10,000 test images, each 28x28 pixels, grayscale, labeled with one of 10 classes.

In [ ]:
# Load Fashion-MNIST. Keras caches it locally after the first download, so re-running this is fast.
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Class index -> human-readable name. Order matters, it must match the dataset's label encoding.
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print("Training images shape  :", x_train.shape)
print("Training labels shape  :", y_train.shape)
print("Test images shape      :", x_test.shape)
print("Test labels shape      :", y_test.shape)
print("Pixel value range      : [{}, {}]".format(x_train.min(), x_train.max()))

## 2. Preprocessing

Two things need to happen before the data goes into a CNN:

1. **Normalization** - raw pixel values are integers in [0, 255]. Neural networks train more stably on small, centered ranges, so we scale everything to [0, 1].
2. **Reshaping** - Keras images load as `(samples, 28, 28)`, but `Conv2D` layers expect an explicit channel dimension, even for grayscale. So the shape becomes `(samples, 28, 28, 1)`.

Yahan pe channel dimension add karna zaroori hai, warna Conv2D layer error de ga (it expects a 4D tensor, not 3D).

In [ ]:
# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Add the channel dimension required by Conv2D: (samples, 28, 28, 1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

print("After reshaping:")
print("  x_train:", x_train.shape)
print("  x_test :", x_test.shape)
print("Pixel value range after normalization: [{:.1f}, {:.1f}]".format(
    x_train.min(), x_train.max()))

## 3. Visualizing Sample Images

Before building the model, it's worth actually looking at the data. Sanity-checking labels here also catches dataset bugs early (e.g. mismatched labels), rather than discovering them after a long training run.

In [ ]:
# Plot a 2x5 grid of training samples with their true labels
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Fashion-MNIST Images', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(class_names[y_train[i]], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('task1_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample images saved as task1_sample_images.png")

## 4. Building the CNN

Architecture: three convolutional blocks of increasing filter depth (32 -> 64 -> 128), each followed by dropout for regularization, then a dense classifier head.

**Why this design:**
- Increasing filter counts let early layers learn simple edges/textures while deeper layers learn more abstract shape patterns (collar shapes, sleeve outlines, etc).
- `padding='same'` keeps spatial dimensions predictable after each conv layer, only `MaxPooling2D` reduces resolution.
- Dropout after every block (0.25) plus a heavier dropout before the final dense layer (0.5) controls overfitting, since Fashion-MNIST is a relatively small dataset for a CNN with over a million parameters.
- Two `MaxPooling2D` layers (not three) keep the spatial map from shrinking too aggressively, the third conv block works on a 7x7 feature map directly.

This is a fairly standard architecture for small grayscale image classification, nothing exotic, deliberately so. The point of this notebook is a solid, explainable baseline rather than a state-of-the-art result.

In [ ]:
model = keras.Sequential([

    # Block 1: learns low-level features (edges, simple textures)
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  input_shape=(28, 28, 1), name='conv1'),
    layers.MaxPooling2D(pool_size=(2, 2), name='pool1'),
    layers.Dropout(0.25, name='dropout1'),

    # Block 2: learns mid-level patterns (shapes, contours)
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
    layers.MaxPooling2D(pool_size=(2, 2), name='pool2'),
    layers.Dropout(0.25, name='dropout2'),

    # Block 3: learns higher-level, more abstract features. No pooling here,
    # the 7x7 feature map is already small enough going into the dense layers.
    layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
    layers.Dropout(0.25, name='dropout3'),

    # Classifier head
    layers.Flatten(name='flatten'),
    layers.Dense(256, activation='relu', name='dense1'),
    layers.Dropout(0.5, name='dropout4'),
    layers.Dense(10, activation='softmax', name='output')
], name='Fashion_CNN')

model.summary()

## 5. Compiling the Model

- **Optimizer:** Adam, the standard default choice, adapts the learning rate per parameter and converges reliably without much manual tuning.
- **Loss:** `sparse_categorical_crossentropy`, used because the labels are integers (0-9) rather than one-hot vectors. Using plain `categorical_crossentropy` here would require one-hot encoding the labels first, which is an unnecessary extra step.
- **Metric:** accuracy, the natural choice for a balanced multi-class classification problem (each class has exactly 1000 test samples, so accuracy is not misleading here the way it can be on imbalanced datasets).

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")
print("  Optimizer  : Adam")
print("  Loss       : Sparse Categorical Crossentropy")
print("  Metric     : Accuracy")

## 6. Training

Training is capped at 20 epochs, but `EarlyStopping` watches validation accuracy and stops once it stops improving for 4 consecutive epochs, restoring the best weights seen so far. This protects against overfitting without having to guess the right epoch count in advance.

15% of the training set is held out as a validation split, used only to monitor generalization during training. It never touches the test set.

In [ ]:
EPOCHS           = 20
BATCH_SIZE        = 64
VALIDATION_SPLIT  = 0.15   # 15% of training data held out for validation

# Stop training once val_accuracy plateaus, restore the best-performing weights
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=[early_stop],
    verbose=1
)

## 7. Training Summary Table

A quick per-epoch table is easier to scan than the verbose training logs above.

In [ ]:
header = f"{'Epoch':>6} {'Train Loss':>12} {'Train Acc':>11} {'Val Loss':>10} {'Val Acc':>9}"
print("\n" + header)
print("-" * 55)
for epoch in range(len(history.history['loss'])):
    print(f"{epoch+1:>6} "
          f"{history.history['loss'][epoch]:>12.4f} "
          f"{history.history['accuracy'][epoch]:>11.4f} "
          f"{history.history['val_loss'][epoch]:>10.4f} "
          f"{history.history['val_accuracy'][epoch]:>9.4f}")

## 8. Evaluating on the Test Set

This is the number that matters most: performance on data the model has never seen during training or validation.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print("="*40)
print("       TEST SET PERFORMANCE")
print("="*40)
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_accuracy*100:.2f}%")
print("="*40)

## 9. Confusion Matrix

Overall accuracy hides which specific classes the model struggles with. The confusion matrix shows that directly: rows are true labels, columns are predicted labels, and the diagonal is where predictions match reality.

In [ ]:
y_pred_probs = model.predict(x_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix \u2014 Fashion-MNIST CNN', fontsize=13, fontweight='bold')
plt.ylabel('True Label',    fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('task4_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved as task4_confusion_matrix.png")

## 10. Classification Report

Per-class precision, recall, and F1-score. This is where the model's actual weak point shows up clearly: the **Shirt** class. Recall for Shirt is noticeably lower than every other class, meaning the model frequently mistakes real shirts for something else (mainly T-shirt/top and Coat, as seen in the confusion matrix above).

In [ ]:
print("\nClassification Report:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=class_names))

## 11. Training Curves

Plotting loss and accuracy across epochs makes it easy to check for overfitting: if training accuracy keeps climbing while validation accuracy flattens or drops, that's the signal. Here validation accuracy actually tracks slightly above training accuracy for most of training, which is expected behavior with dropout active (dropout makes the training metric a bit noisier/lower than the validation metric, since dropout is disabled during validation).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Training History', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['loss']) + 1)

# Loss curve
axes[0].plot(epochs_range, history.history['loss'],     label='Training Loss',   color='#E74C3C', linewidth=2)
axes[0].plot(epochs_range, history.history['val_loss'], label='Validation Loss', color='#3498DB', linewidth=2, linestyle='--')
axes[0].set_title('Loss per Epoch',     fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, history.history['accuracy'],     label='Training Accuracy',   color='#27AE60', linewidth=2)
axes[1].plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy', color='#8E44AD', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy per Epoch', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task5_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved as task5_training_curves.png")

## 12. Sample Predictions

A random sample of test images with predicted vs true labels, color-coded green for correct and red for incorrect. Seeing actual misclassified images (rather than just numbers in a table) makes it much easier to understand why the model fails on certain inputs, e.g. a sleeveless dark top genuinely does look ambiguous between Shirt and T-shirt/top even to a human eye.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 9))
fig.suptitle('Test Images: Predicted vs True Labels\n(Green = Correct, Red = Incorrect)',
             fontsize=13, fontweight='bold')

# Fixed seed so the same 15 images are picked every time this cell runs
np.random.seed(42)
indices = np.random.choice(len(x_test), 15, replace=False)

for i, (ax, idx) in enumerate(zip(axes.flat, indices)):
    ax.imshow(x_test[idx].reshape(28, 28), cmap='gray')

    true_label  = class_names[y_test[idx]]
    pred_label  = class_names[y_pred[idx]]
    color       = 'green' if y_test[idx] == y_pred[idx] else 'red'

    ax.set_title(f'Pred: {pred_label}\nTrue: {true_label}',
                 fontsize=7.5, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('task5_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Predictions grid saved as task5_predictions.png")

## 13. Final Summary

In [ ]:
print("\n" + "="*50)
print("           FINAL MODEL SUMMARY")
print("="*50)
print(f"  Dataset         : Fashion-MNIST")
print(f"  Architecture    : CNN (3 Conv + 4 Dropout + Dense)")
print(f"  Total Epochs    : {len(history.history['loss'])}")
print(f"  Test Accuracy   : {test_accuracy*100:.2f}%")
print(f"  Test Loss       : {test_loss:.4f}")
best_val = max(history.history['val_accuracy'])
print(f"  Best Val Acc    : {best_val*100:.2f}%")
print("="*50)

## 14. Interactive Demo (Gradio)

A small web UI for testing the trained model on your own images, useful for quick manual sanity checks beyond the fixed test set, and far easier to show in a demo or viva than reading numbers off a table.

**Important limitation to keep in mind while testing:** this model was trained only on Fashion-MNIST, meaning 28x28 grayscale images of a single clothing item, roughly centered, on a plain dark background. It was never shown a real photograph, a colour image, multiple objects in frame, or a cluttered background. Uploading a normal phone photo of a shirt on a table will get resized down to 28x28 and converted to grayscale before prediction, but the result is likely to be unreliable, since that input does not resemble the training distribution at all. This is not a bug in the interface, it is an inherent limitation of the model itself, and a fair point to mention in your report under "real-world limitations": low-resolution academic datasets like Fashion-MNIST do not transfer well to unconstrained real-world images without retraining or fine-tuning on a more representative dataset.

For the most meaningful test results, try cropping an image down to roughly square, with the clothing item filling most of the frame, similar in spirit to the dataset samples shown earlier in this notebook.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
from PIL import Image


def preprocess_uploaded_image(pil_image):
    """
    Convert an arbitrary uploaded image into the exact format the model expects:
    grayscale, 28x28, normalized to [0, 1], shaped (1, 28, 28, 1).

    Fashion-MNIST images have a dark background with a light-colored garment,
    so if the uploaded image looks like the opposite (light background, dark
    subject), colors are inverted to roughly match what the model was trained on.
    This is a heuristic, not a guarantee, real photos vary a lot more than the
    dataset ever did.
    """
    grayscale = pil_image.convert('L')
    resized = grayscale.resize((28, 28))

    array = np.array(resized).astype('float32')

    # Heuristic background check: Fashion-MNIST backgrounds are dark (near 0),
    # subjects are lighter. If the uploaded image is mostly bright (e.g. a photo
    # on a white background), invert it so the model sees a familiar pattern.
    if array.mean() > 127:
        array = 255.0 - array

    array = array / 255.0
    array = array.reshape(1, 28, 28, 1)
    return array, resized


def predict_clothing_item(pil_image):
    if pil_image is None:
        return {}, None

    model_input, preview_image = preprocess_uploaded_image(pil_image)
    probabilities = model.predict(model_input, verbose=0)[0]

    # Gradio's Label component expects a dict of {class_name: confidence}
    confidence_scores = {
        class_names[i]: float(probabilities[i]) for i in range(len(class_names))
    }

    # Also return the exact 28x28 grayscale version the model actually saw,
    # upscaled for visibility, this makes it obvious when preprocessing has
    # destroyed too much detail for the prediction to be trustworthy.
    preview_upscaled = preview_image.resize((140, 140), Image.NEAREST)

    return confidence_scores, preview_upscaled


demo = gr.Interface(
    fn=predict_clothing_item,
    inputs=gr.Image(type='pil', label='Upload a clothing image'),
    outputs=[
        gr.Label(num_top_classes=5, label='Predicted class (top 5)'),
        gr.Image(label='What the model actually saw (28x28 grayscale input)')
    ],
    title='Fashion-MNIST CNN Classifier',
    description=(
        'Upload an image of a single clothing item. Works best with a plain '
        'background and the item filling most of the frame, similar to the '
        'Fashion-MNIST training images. Real-world photos with cluttered '
        'backgrounds or multiple objects are outside this model\'s training '
        'distribution and may produce unreliable predictions.'
    ),
    examples=None,
    allow_flagging='never'
)

demo.launch(debug=False)

## Notes and Limitations

- Training stopped at epoch 5 because of early stopping. This is normal behavior, not a bug, validation accuracy stopped improving so training halted to avoid overfitting and wasted compute.
- The weakest class is **Shirt** (recall 0.66), which gets confused mainly with **T-shirt/top** and **Coat**. This is a known difficulty in Fashion-MNIST: these categories share very similar silhouettes at 28x28 resolution, even human annotators occasionally disagree on these boundaries.
- No data augmentation (rotation, shift, zoom) was used in this version. Adding it would likely help generalization slightly further, particularly for the confused classes, at the cost of longer training time.
- No hyperparameter search was performed, filter counts, dropout rates, and dense layer size were chosen based on common practice for this dataset size, not tuned via grid/random search.
- The Gradio demo's background-inversion heuristic (inverting colors when the uploaded image is mostly bright) is a rough fix for one specific mismatch between real photos and Fashion-MNIST's dark-background convention. It does not solve the deeper issue that this model has never seen a real photograph, a colour image, or a cluttered background during training, so predictions on arbitrary uploaded images should be treated as a fun sanity check, not a reliable result.